# Advanced Data Visualization

**Primary outcome:** Build statistical visualizations with Seaborn and interactive charts with Plotly, using real-world datasets.

**Estimated time:** 60–90 minutes

---

## What you will learn

| Section | Tool | Skills |
|---------|------|---------|
| Part 1 | Seaborn | Distribution plots, categorical plots, relationship plots, heatmaps |
| Part 2 | Plotly | Interactive scatter, animations, bar/line charts, choropleth maps |

## Datasets

- **Penguins** — physical measurements of 344 penguins across 3 species (Seaborn built-in)
- **Flights** — monthly airline passenger counts 1949–1960 (Seaborn built-in)
- **Gapminder** — country-level life expectancy, GDP, and population 1952–2007 (Plotly built-in)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Global style for Seaborn
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries loaded successfully.')

---

## Part 1: Seaborn — Statistical Visualization

---

### 1. Introduction to Seaborn

Matplotlib gives you fine-grained control over every element of a figure. Seaborn builds on top of it and adds:

- **Statistical awareness** — it can compute and overlay confidence intervals, kernel density estimates, and regression fits automatically.
- **DataFrame-native API** — pass column names directly instead of extracting arrays.
- **Semantic mappings** — map data variables to `hue`, `size`, and `style` with a single argument.
- **Attractive defaults** — publication-quality themes out of the box.

Use Seaborn when your goal is **exploratory data analysis** or communicating statistical relationships. Reach back to Matplotlib when you need pixel-level layout control.

In [ ]:
# Load the penguins dataset
penguins = sns.load_dataset('penguins')

print('Shape:', penguins.shape)
print('\nColumn types:')
print(penguins.dtypes)
print('\nMissing values:')
print(penguins.isnull().sum())

In [ ]:
# Quick preview
penguins.head()

In [ ]:
# Summary statistics — useful before plotting so you know the range and spread
penguins.describe()

---

### 2. Distribution Plots

Distribution plots answer: **How is this variable spread across its range?**

| Plot | Best when |
|------|-----------|
| `histplot` | You want counts per bin; fast overview |
| `kdeplot` | You want a smooth density curve; good for comparisons |
| `boxplot` | You want median, IQR, and outliers at a glance |
| `violinplot` | You want the full distribution shape per group |

**Rule of thumb:** start with a histogram, then layer on KDE if you want a smoother picture.

In [ ]:
# Histogram with KDE overlay
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=penguins,
    x='body_mass_g',
    kde=True,          # overlay a kernel density estimate
    hue='species',
    bins=25,
    ax=ax
)
ax.set_title('Body Mass Distribution by Species')
ax.set_xlabel('Body Mass (g)')
plt.tight_layout()
plt.show()

In [ ]:
# KDE plot — useful when you want to compare smooth distributions
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(
    data=penguins,
    x='flipper_length_mm',
    hue='species',
    fill=True,          # shade the area under the curve
    alpha=0.4,
    ax=ax
)
ax.set_title('Flipper Length Density by Species')
ax.set_xlabel('Flipper Length (mm)')
plt.tight_layout()
plt.show()

In [ ]:
# Box plot — median, IQR (box), whiskers (1.5 × IQR), and outlier dots
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(
    data=penguins,
    x='species',
    y='body_mass_g',
    hue='species',
    palette='Set2',
    legend=False,
    ax=ax
)
ax.set_title('Body Mass by Species (Box Plot)')
ax.set_ylabel('Body Mass (g)')
plt.tight_layout()
plt.show()

In [ ]:
# Violin plot — combines box plot summary with KDE shape on both sides
fig, ax = plt.subplots(figsize=(8, 4))
sns.violinplot(
    data=penguins,
    x='species',
    y='body_mass_g',
    hue='species',
    palette='Set2',
    inner='quartile',   # draw quartile lines inside the violin
    legend=False,
    ax=ax
)
ax.set_title('Body Mass by Species (Violin Plot)')
ax.set_ylabel('Body Mass (g)')
plt.tight_layout()
plt.show()

**Observation:** Gentoo penguins are significantly heavier than Adelie and Chinstrap. The violin plot reveals that Chinstrap has a slightly bimodal shape, which the box plot alone would not show.

---

### 3. Categorical Plots

Categorical plots compare values or counts across discrete groups.

| Plot | Best when |
|------|-----------|
| `countplot` | Counting occurrences per category |
| `barplot` | Showing aggregated values (mean ± CI) per category |
| `stripplot` | Showing every data point; low overlap |
| `swarmplot` | Showing every data point; avoids overlap by spreading |

`stripplot` is faster for large datasets. `swarmplot` is more informative but slows down above ~5 000 points.

In [ ]:
# Count plot — how many penguins per island?
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(
    data=penguins,
    x='island',
    hue='species',
    palette='Set1',
    ax=ax
)
ax.set_title('Penguin Count per Island by Species')
ax.set_xlabel('Island')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot — mean bill length by species and sex, with 95 % CI error bars
penguins_clean = penguins.dropna(subset=['bill_length_mm', 'sex'])

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(
    data=penguins_clean,
    x='species',
    y='bill_length_mm',
    hue='sex',
    palette='coolwarm',
    capsize=0.1,        # cap width on CI error bars
    ax=ax
)
ax.set_title('Mean Bill Length by Species and Sex (95% CI)')
ax.set_ylabel('Bill Length (mm)')
plt.tight_layout()
plt.show()

In [ ]:
# Strip plot vs Swarm plot — side by side comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

sns.stripplot(
    data=penguins,
    x='species',
    y='bill_depth_mm',
    hue='species',
    jitter=True,        # add horizontal noise to reduce overplotting
    alpha=0.6,
    legend=False,
    ax=axes[0]
)
axes[0].set_title('Strip Plot — bill depth by species')
axes[0].set_ylabel('Bill Depth (mm)')

sns.swarmplot(
    data=penguins,
    x='species',
    y='bill_depth_mm',
    hue='species',
    size=4,
    legend=False,
    ax=axes[1]
)
axes[1].set_title('Swarm Plot — bill depth by species')
axes[1].set_ylabel('')

plt.suptitle('Strip vs Swarm: every point is plotted', y=1.02)
plt.tight_layout()
plt.show()

**Takeaway:** Both strip and swarm plots show every individual observation. Strip plots randomly jitter points horizontally; swarm plots arrange them in a bee-swarm pattern so no two points overlap. Swarm plots are more honest but computationally expensive at scale.

---

### 4. Relationship Plots

Relationship plots reveal how two (or more) numeric variables move together.

- **`scatterplot`** — two variables, with optional hue/size/style encodings for extra dimensions.
- **`pairplot`** — a grid of scatter plots for every pair of numeric variables; diagonal shows distributions. Excellent for a **multivariate overview at a glance**.

In [ ]:
# Scatter plot — bill length vs bill depth, coloured by species
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=penguins,
    x='bill_length_mm',
    y='bill_depth_mm',
    hue='species',
    style='species',    # different marker per species for accessibility
    s=80,
    alpha=0.8,
    ax=ax
)
ax.set_title('Bill Length vs Bill Depth by Species')
ax.set_xlabel('Bill Length (mm)')
ax.set_ylabel('Bill Depth (mm)')
plt.tight_layout()
plt.show()

Notice that at the aggregate level the scatter appears to show a *negative* correlation between bill length and depth. But **within** each species the relationship is actually *positive*. This is an example of **Simpson's Paradox** — the trend reverses when the data is disaggregated.

In [ ]:
# Pair plot — all numeric variables against each other, diagonal = KDE
# Drop rows with missing values first
penguins_clean_all = penguins.dropna()

g = sns.pairplot(
    penguins_clean_all,
    hue='species',
    diag_kind='kde',    # KDE on diagonal instead of histogram
    plot_kws={'alpha': 0.6, 's': 40},
    height=2.2
)
g.fig.suptitle('Pair Plot — All Numeric Features by Species', y=1.02)
plt.show()

**Reading a pair plot:**
- **Off-diagonal cells** — scatter plot of row variable (y-axis) vs column variable (x-axis).
- **Diagonal cells** — distribution of that single variable.
- **Well-separated clusters by colour** → those two variables have strong discriminatory power between groups.

`flipper_length_mm` vs `body_mass_g` shows the cleanest Gentoo separation, making it a candidate for a classification feature.

---

### 5. Heatmap

A heatmap encodes a **2-D matrix** of values as colour intensity. It is ideal for:

- **Time × category frequency tables** — e.g., flights per month per year.
- **Correlation matrices** — how strongly each pair of numeric features is correlated.
- **Confusion matrices** — model classification performance.

We use the **flights** dataset — monthly passenger counts from 1949 to 1960.

In [ ]:
# Load flights dataset and pivot to a year × month matrix
flights = sns.load_dataset('flights')
print('Shape:', flights.shape)
flights.head()

In [ ]:
# Pivot table: rows = year, columns = month, values = number of passengers
flights_pivot = flights.pivot(index='year', columns='month', values='passengers')
flights_pivot

In [ ]:
# Heatmap — annotated with integer passenger counts
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    flights_pivot,
    annot=True,         # show the numeric value in each cell
    fmt='d',            # format as integer (no decimal point)
    cmap='YlOrRd',      # yellow → orange → red = low → high
    linewidths=0.5,
    linecolor='white',
    ax=ax
)
ax.set_title('Monthly Airline Passengers (1949–1960)', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

**Observations from the heatmap:**
1. **Trend over years** — every row gets darker (more passengers) as you move down, showing steady year-on-year growth.
2. **Seasonal pattern** — July and August are consistently the busiest months (darkest in each row), while November and February are the quietest.
3. **Interaction effect** — the absolute seasonal difference also grows over time (the summer–winter gap widens), visible as the contrast between light and dark cells increasing row by row.

---

## Part 2: Plotly — Interactive Visualization

---

### 6. Introduction to Plotly

Seaborn produces **static** images. Once rendered, the reader cannot explore the data further. Plotly produces **interactive** HTML-based charts where the reader can:

- **Hover** to see exact values.
- **Zoom and pan** to inspect dense regions.
- **Toggle** legend entries to isolate groups.
- **Play animations** to watch data change over time.

| Use case | Better tool |
|----------|-------------|
| Static report / paper / slide | Seaborn / Matplotlib |
| Web dashboard / notebook exploration | Plotly |
| Presentations with time-series storytelling | Plotly animations |
| Geographic data | Plotly choropleth |

`plotly.express` (imported as `px`) is the high-level API — one function call per chart type.

In [ ]:
# Load Gapminder and inspect
gapminder = px.data.gapminder()
print('Shape:', gapminder.shape)
print('\nColumns:', gapminder.columns.tolist())
print('\nYears available:', sorted(gapminder['year'].unique()))
gapminder.head()

In [ ]:
# Filter to 2007 for cross-sectional analysis
gapminder_2007 = gapminder[gapminder['year'] == 2007].copy()
print('Countries in 2007 snapshot:', len(gapminder_2007))

---

### 7. Interactive Scatter Plot

We map four variables simultaneously:
- **x-axis** — GDP per capita (log scale)
- **y-axis** — life expectancy
- **bubble size** — population
- **colour** — continent

Hover over any bubble to see the country name and exact values. Use the toolbar (top-right) to zoom or pan.

In [ ]:
fig = px.scatter(
    gapminder_2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    color='continent',
    hover_name='country',       # shown in bold at top of tooltip
    hover_data=['pop', 'gdpPercap', 'lifeExp'],
    log_x=True,                 # log scale reveals the spread better
    size_max=60,                # cap maximum bubble size
    title='GDP per Capita vs Life Expectancy (2007)',
    labels={
        'gdpPercap': 'GDP per Capita (USD, log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'pop': 'Population'
    }
)
fig.update_layout(height=550)
fig.show()

**Try it yourself:** Double-click on "Asia" in the legend to isolate Asian countries. Notice the large bubbles for China and India, and the high life expectancy of Japan despite not being the wealthiest.

---

### 8. Animated Scatter Chart

The same bubble chart, but animated across all years from 1952 to 2007. This is one of the most compelling ways to tell a data story — the audience watches countries develop in real time.

Press the **Play** button below the chart to start the animation.

In [ ]:
fig_anim = px.scatter(
    gapminder,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    color='continent',
    hover_name='country',
    animation_frame='year',         # create one frame per year
    animation_group='country',      # track the same country across frames
    log_x=True,
    size_max=55,
    range_x=[100, 100_000],         # fix axis range so it doesn't rescale
    range_y=[25, 90],
    title='GDP per Capita vs Life Expectancy — Animated (1952–2007)',
    labels={
        'gdpPercap': 'GDP per Capita (USD, log scale)',
        'lifeExp': 'Life Expectancy (years)'
    }
)
fig_anim.update_layout(height=580)
fig_anim.show()

**What to look for:**
- The overall cloud moves **up and to the right** over time — most countries got richer and healthier.
- Some African countries show stagnation or temporary drops in life expectancy in the 1990s, coinciding with the HIV/AIDS epidemic.
- The speed of East Asian development (South Korea, China) is especially visible.

> **Animations work best for time-series storytelling** when you want your audience to *feel* the change rather than just see a static before/after.

---

### 9. Interactive Bar Chart and Line Chart

In [ ]:
# Bar chart — top 10 countries by GDP per capita in 2007
top10_gdp = (
    gapminder_2007
    .nlargest(10, 'gdpPercap')
    .sort_values('gdpPercap', ascending=True)  # ascending so highest is at top
)

fig_bar = px.bar(
    top10_gdp,
    x='gdpPercap',
    y='country',
    color='continent',
    orientation='h',                # horizontal bars are easier to read for long labels
    hover_data=['lifeExp', 'pop'],
    title='Top 10 Countries by GDP per Capita (2007)',
    labels={'gdpPercap': 'GDP per Capita (USD)', 'country': ''}
)
fig_bar.update_layout(height=420)
fig_bar.show()

In [ ]:
# Line chart — life expectancy over time for selected countries
countries_of_interest = ['Australia', 'Singapore', 'Japan', 'United Kingdom']
selected = gapminder[gapminder['country'].isin(countries_of_interest)]

fig_line = px.line(
    selected,
    x='year',
    y='lifeExp',
    color='country',
    markers=True,               # show data points on the line
    hover_data=['gdpPercap'],
    title='Life Expectancy Over Time — Australia, Singapore, Japan, UK',
    labels={'lifeExp': 'Life Expectancy (years)', 'year': 'Year'}
)
fig_line.update_layout(height=450)
fig_line.show()

**Note:** Singapore starts well below the others in 1952 but ends near the top by 2007 — one of the fastest life expectancy improvements in the dataset. Hover over the Singapore line to confirm the exact values.

---

### 10. Choropleth Map

A **choropleth map** shades geographic regions (countries, states, districts) by a data variable. It is the natural choice when:

- Your data has a **geographic dimension** (country codes, state abbreviations, lat/lon).
- You want to reveal **spatial patterns** — which regions lead or lag on a metric.

Gapminder includes `iso_alpha` (ISO 3166-1 alpha-3 country codes) which Plotly uses to match countries to map polygons automatically.

In [ ]:
fig_map = px.choropleth(
    gapminder_2007,
    locations='iso_alpha',              # column containing ISO country codes
    color='lifeExp',
    hover_name='country',
    hover_data=['gdpPercap', 'pop'],
    color_continuous_scale='Viridis',   # perceptually uniform colour scale
    range_color=(30, 85),               # pin the colour range
    title='Life Expectancy by Country (2007)',
    labels={'lifeExp': 'Life Expectancy (years)'}
)
fig_map.update_layout(
    geo=dict(showframe=False, showcoastlines=True),
    height=520
)
fig_map.show()

**Tips for choropleth maps:**
- Choose a **perceptually uniform** colour scale (Viridis, Plasma, Cividis) so the visual difference matches the data difference.
- Avoid rainbow scales (Jet) — they create false visual boundaries.
- For diverging data (e.g., change from a baseline), use a diverging scale (RdBu, RdYlGn) centred at zero.

---

## Practice Exercises

Apply what you have learned. Attempt each exercise before expanding the solution.

---

### Exercise 1 — Seaborn Distribution

Plot a **violin plot** of `flipper_length_mm` from the penguins dataset, split by `sex` (x-axis) and coloured by `species`. Drop rows with missing `sex` values first.

**Bonus:** Add a box plot overlay inside the violin using `inner='box'`.

In [ ]:
# Your code here


<details>
<summary><strong>Solution — click to expand</strong></summary>

```python
penguins_no_na = penguins.dropna(subset=['sex', 'flipper_length_mm'])

fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(
    data=penguins_no_na,
    x='sex',
    y='flipper_length_mm',
    hue='species',
    inner='box',
    palette='Set2',
    ax=ax
)
ax.set_title('Flipper Length by Sex and Species')
ax.set_ylabel('Flipper Length (mm)')
plt.tight_layout()
plt.show()
```

</details>

---

### Exercise 2 — Seaborn Heatmap

Using the **penguins** dataset (drop rows with `NaN`), compute the **Pearson correlation matrix** of all four numeric columns (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`) and plot it as an annotated heatmap.

**Hint:** `df.corr()` returns the correlation matrix. Use `cmap='coolwarm'` and `vmin=-1, vmax=1` to centre the colour scale at zero.

In [ ]:
# Your code here


<details>
<summary><strong>Solution — click to expand</strong></summary>

```python
numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
corr_matrix = penguins[numeric_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Penguin Numeric Feature Correlation Matrix')
plt.tight_layout()
plt.show()
```

**Interpretation:** `flipper_length_mm` and `body_mass_g` have the highest positive correlation (~0.87), suggesting heavier penguins have longer flippers. `bill_depth_mm` is *negatively* correlated with `bill_length_mm` at the species-aggregated level due to Simpson's Paradox.

</details>

---

### Exercise 3 — Plotly Interactive

Using the full **gapminder** dataset (all years), create a **line chart** showing **GDP per capita over time** for the five most populous countries in 2007.

**Steps:**
1. Find the top 5 countries by population in 2007.
2. Filter the full `gapminder` DataFrame to those countries.
3. Plot with `px.line()`, colouring by country.

**Bonus:** Add `markers=True` and include `lifeExp` in `hover_data`.

In [ ]:
# Your code here


<details>
<summary><strong>Solution — click to expand</strong></summary>

```python
# Step 1 — identify top 5 by population in 2007
top5_countries = (
    gapminder_2007
    .nlargest(5, 'pop')['country']
    .tolist()
)
print('Top 5 most populous in 2007:', top5_countries)

# Step 2 — filter all years
top5_df = gapminder[gapminder['country'].isin(top5_countries)]

# Step 3 — plot
fig = px.line(
    top5_df,
    x='year',
    y='gdpPercap',
    color='country',
    markers=True,
    hover_data=['lifeExp', 'pop'],
    title='GDP per Capita Over Time — 5 Most Populous Countries (2007)',
    labels={'gdpPercap': 'GDP per Capita (USD)', 'year': 'Year'}
)
fig.update_layout(height=480)
fig.show()
```

</details>

---

## Summary

### Seaborn — Statistical Visualization

| Plot | Function | Key parameters |
|------|----------|----------------|
| Histogram + KDE | `sns.histplot` | `kde=True`, `hue`, `bins` |
| Density curve | `sns.kdeplot` | `fill=True`, `hue` |
| Box plot | `sns.boxplot` | `x`, `y`, `hue` |
| Violin plot | `sns.violinplot` | `inner='quartile'` or `'box'` |
| Count plot | `sns.countplot` | `x`, `hue` |
| Bar plot (mean ± CI) | `sns.barplot` | `capsize`, `hue` |
| Strip plot | `sns.stripplot` | `jitter=True`, `alpha` |
| Swarm plot | `sns.swarmplot` | `size` |
| Scatter | `sns.scatterplot` | `hue`, `style`, `size` |
| Pair plot | `sns.pairplot` | `hue`, `diag_kind` |
| Heatmap | `sns.heatmap` | `annot`, `fmt`, `cmap` |

### Plotly — Interactive Visualization

| Chart | Function | When to use |
|-------|----------|-------------|
| Bubble scatter | `px.scatter` | Compare 4 variables + hover |
| Animated scatter | `px.scatter` + `animation_frame` | Time-series storytelling |
| Horizontal bar | `px.bar` | Ranked comparisons |
| Line chart | `px.line` | Trends over time |
| Choropleth map | `px.choropleth` | Geographic patterns |

### Choosing the right tool

- **Seaborn** for statistical analysis and static reports — distributions, relationships, aggregates with confidence intervals.
- **Plotly** for interactive exploration, web dashboards, and presentations — hover, zoom, filter, and animate.
- Neither tool is universally better. In practice, use both: Seaborn for EDA, Plotly for sharing results.